# Stochastic Calculus, Step by Step · Lesson 27: Change of numéraire

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Counting in shares instead of dollars · 3. The share-measure weight · 4. Prices in shares are martingales · 5. What N(d+) really is · 6. A price with no rate in it · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 27,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Counting in shares instead of dollars

A **numéraire** is the unit you count wealth in. So far it has been the cash account, $e^{rt}$, and $\tilde P$ is the probability that makes everything counted in cash a martingale. Choose the share as the unit instead. Then cash is a risky asset and the share is the safe one: the share is worth exactly 1 share, forever.

One outcome in both units:

In [ ]:
rng = np.random.default_rng(seed)
M_ = 2_000_000
Zv = rng.standard_normal(M_)
S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Zv)
disc = math.exp(-r * T)
one = {"S_T": 120.0, "pay": 15.0, "pay_shares": 15.0 / 120.0, "cash_T": math.exp(r * T), "cash_in_shares": math.exp(r * T) / 120.0, "cash0_in_shares": 1 / S0}
out["unit"] = one
one

## 3. The share-measure weight

Define $P^S$ by the weight $Z^S_T = \dfrac{e^{-rT}S_T}{S_0}$ on $\tilde P$. It is positive, with average 1 because the discounted share is a $\tilde P$-martingale. Writing it out,

$$Z^S_T = e^{\sigma\tilde W_T - \frac12\sigma^2T},$$

the Lesson 25 weight with $\theta = -\sigma$. So Girsanov says: under $P^S$, $W^S_t = \tilde W_t - \sigma t$ is a Brownian motion. The share's drift rises by $\sigma^2$: $dS = (r + \sigma^2)S\,dt + \sigma S\,dW^S$.

In [ ]:
ZS = disc * S_T / S0
ES = lambda x: float((ZS * x).mean())
out["weight"] = {"mean": float(ZS.mean()), "var": float(ZS.var()), "var_exact": math.exp(sigma**2 * T) - 1,
                 "mean_W_S": ES(math.sqrt(T) * Zv), "mean_W_S_exact": sigma * T, "var_W": ES(T * Zv**2) - ES(math.sqrt(T) * Zv) ** 2,
                 "mean_ln": ES(np.log(S_T / S0)), "mean_ln_exact": (r + 0.5 * sigma**2) * T, "mean_ln_Q": float(np.log(S_T / S0).mean()),
                 "median_Q": S0 * math.exp((r - 0.5 * sigma**2) * T), "median_S": S0 * math.exp((r + 0.5 * sigma**2) * T)}
out["weight"]

## 4. Prices in shares are martingales

Under $P^S$, anything counted in shares is a martingale. Cash counted in shares is $e^{rt}/S_t$, and its average stays at $1/S_0$. The option counted in shares has the same property, which gives a second pricing formula:

$$\frac{V_0}{S_0} = E^S\Big[\frac{V_T}{S_T}\Big] .$$

In [ ]:
# simulate directly under the share measure, with drift r + sigma^2
gen = np.random.default_rng(seed + 1)
Zs = gen.standard_normal(M_)
S_S = S0 * np.exp((r + 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Zs)
ratio_pay = S0 * np.maximum(S_S - K, 0) / S_S
mart_rows = []
for t_ in (0.25, 0.5, 1.0):
    S_t = S0 * np.exp((r + 0.5 * sigma**2) * t_ + sigma * math.sqrt(t_) * Zs)
    mart_rows.append({"t": t_, "mean_ratio": float((math.exp(r * t_) / S_t).mean()), "target": 1 / S0})
out["mart"] = {"rows": mart_rows, "call_S": float(ratio_pay.mean()), "call_S_se": float(ratio_pay.std() / math.sqrt(M_)),
               "call_Q": float(disc * np.maximum(S_T - K, 0).mean()), "call_Q_se": float(disc * np.maximum(S_T - K, 0).std() / math.sqrt(M_)), "call_bs": float(call(S0, T))}
pd.DataFrame(mart_rows)

## 5. What N(d+) really is

Split the call into two payoffs, $S_T\mathbf 1\{S_T > K\}$ and $K\mathbf 1\{S_T > K\}$. Price each in the unit that suits it:

$$C = S_0\,P^S(S_T > K) - Ke^{-rT}\,\tilde P(S_T > K) = S_0N(d_+) - Ke^{-rT}N(d_-) .$$

So $N(d_-)$ and $N(d_+)$ are both the probability that the call finishes in the money, each in its own unit: cash and shares. The second is higher, because share units favour paths where the share ends high.

In [ ]:
sg = np.linspace(40, 220, 361)
dens = lambda s, m_: phi((np.log(s / S0) - (m_ - 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))) / (s * sigma * math.sqrt(T))
out["probs"] = {"P_Q": float((S_T > K).mean()), "P_S": float((S_S > K).mean()), "N_minus": N(float(d_minus(S0, T))), "N_plus": N(float(d_plus(S0, T))),
                "P_S_w": ES(S_T > K), "gap": N(float(d_plus(S0, T))) - N(float(d_minus(S0, T))), "fig": {"S": sg.tolist(), "Q": dens(sg, r).tolist(), "S_m": dens(sg, r + sigma**2).tolist()},
                "term1": S0 * N(float(d_plus(S0, T))), "term2": K * disc * N(float(d_minus(S0, T))), "P_real": N((math.log(S0 / K) + (mu - 0.5 * sigma**2) * T) / sigma)}
{k_: v for k_, v in out["probs"].items() if k_ != "fig"}

## 6. A price with no rate in it

An **exchange option** pays $\max(S^1_T - S^2_T, 0)$: at $T$, swap one share of asset 2 for one of asset 1. Count everything in shares of asset 2, and the ratio $S^1/S^2$ is a martingale with volatility $\sigma_x = \sqrt{\sigma_1^2 + \sigma_2^2 - 2\rho\sigma_1\sigma_2}$. It is a Black-Scholes call with strike 1 and a zero interest rate (**Margrabe's formula**):

$$V_0 = S^1_0N(d_1) - S^2_0N(d_2), \qquad d_{1,2} = \frac{\ln(S^1_0/S^2_0) \pm \frac12\sigma_x^2T}{\sigma_x\sqrt T} .$$

The cash rate $r$ doesn't appear.

In [ ]:
def margrabe(a0, b0, s_a, s_b, rho, t_=T):
    vx = math.sqrt(s_a**2 + s_b**2 - 2 * rho * s_a * s_b)
    d1 = (math.log(a0 / b0) + 0.5 * vx**2 * t_) / (vx * math.sqrt(t_)); d2 = d1 - vx * math.sqrt(t_)
    return a0 * N(d1) - b0 * N(d2), vx


a0, b0, s_a, s_b, rho_ = 100.0, 95.0, 0.20, 0.30, 0.5
price, vx = margrabe(a0, b0, s_a, s_b, rho_)
gen = np.random.default_rng(seed + 2)
z1 = gen.standard_normal(M_); z2 = rho_ * z1 + math.sqrt(1 - rho_**2) * gen.standard_normal(M_)
mar_rows = []
for rate in (0.0, 0.03, 0.10):
    A = a0 * np.exp((rate - 0.5 * s_a**2) * T + s_a * math.sqrt(T) * z1)
    B = b0 * np.exp((rate - 0.5 * s_b**2) * T + s_b * math.sqrt(T) * z2)
    y_ = math.exp(-rate * T) * np.maximum(A - B, 0)
    mar_rows.append({"r_pct": rate * 100, "mc": float(y_.mean()), "se": float(y_.std() / math.sqrt(M_))})
out["margrabe"] = {"price": price, "vx": vx, "rows": mar_rows, "a0": a0, "b0": b0, "s_a": s_a, "s_b": s_b, "rho": rho_}
pd.DataFrame(mar_rows)

In [ ]:
# Chart data for the video (no random numbers)
out["probs"]["fig"]["ticks"] = [{"x": float(x_), "label": f"{x_:.0f}"} for x_ in (60, 100, 140, 180, 220)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Show that $Z^S_T = e^{-rT}S_T/S_0$ has $\tilde E[Z^S_T] = 1$, in two ways: by the martingale property of the discounted share, and by writing it as an exponential of a normal.

**Exercise 2.** Find the law of $S_T$ under $P^S$. Using the MGF and the weight, show that $\ln S_T$ under $P^S$ is normal with mean $\ln S_0 + (r + \frac12\sigma^2)T$ and variance $\sigma^2T$. What is the median of $S_T$ under $P^S$ with the course's numbers?

**Exercise 3.** Use Itô's lemma on $Y_t = e^{rt}/S_t$ with $dS = (r + \sigma^2)S\,dt + \sigma S\,dW^S$. Show that the $dt$ term vanishes, so cash counted in shares is a martingale under $P^S$.

**Exercise 4.** Derive $P^S(S_T > K) = N(d_+)$ directly, by standardising $\ln S_T$ under $P^S$. Then write $C$ as the sum of two terms, one priced with each unit, and explain in words why $N(d_+) > N(d_-)$.

**Exercise 5.** Derive Margrabe's formula: count in units of asset 2, use the weight $Z = e^{-rT}S^2_T/S^2_0$, find the ratio's volatility, and read the answer off the Black-Scholes formula with $r = 0$ and strike 1. Evaluate it for $S^1_0 = 100$, $S^2_0 = 95$, $\sigma_1 = 20\%$, $\sigma_2 = 30\%$, $\rho = 0.5$, $T = 1$.

**Exercise 6.** Do the same split for the put, $P = Ke^{-rT}N(-d_-) - S_0N(-d_+)$. Read each $N(\cdot)$ as a probability in its own unit. Explain why the put's two probabilities are of finishing **below** the strike.

**Exercise 7.** (Code) Estimate $N(d_+)$ two ways: by simulating under $P^S$ directly, and by weighting $\tilde P$ paths with $Z^S_T$. Which has the smaller standard error? Why? Try $K = 105$ and $K = 160$.

**Exercise 8.** (Code) Check Margrabe's formula by Monte Carlo for $\rho$ of $-0.5, 0, 0.5, 0.9$, using correlated normals. How does the price depend on $\rho$, and why? Check too that changing $r$ from 0 to 10% changes nothing.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")